# Notebook 06 — Fraud Detection Platform: Model Tiering Matrix
**Master Playbook / gap-analysis priority 2 (critical) — classifies the real champion model's materiality (Financial Exposure, Regulatory Scrutiny, Customer Impact) using a transparent, disclosed-ASSUMPTION point rubric applied to real numbers already computed by NB1, NB2, NB3, NB4 and NB5. No recomputation, no retraining, no new model scoring — pure aggregation, and the leanest notebook in this repo (no CSV load, no ML imports).**


In [ ]:
# ============================================================
# SETUP -- WARP-optimized environment. This notebook does no ML work (no
# CSV load, no model load) -- it aggregates real numbers already computed
# by NB1/NB2/NB3/NB4/NB5, so the thread ceiling below is set for
# consistency with every other notebook in this repo, not because this one
# needs it.
# ============================================================
import os, time, json, warnings, subprocess, sys
from datetime import datetime, timezone
warnings.filterwarnings("ignore")

_RUN_T0 = time.time()

CPU_THRESHOLD_PCT = 93
RAM_THRESHOLD_PCT = 90

_N_THREADS = max(1, int((os.cpu_count() or 4) * (CPU_THRESHOLD_PCT / 100) // 1))
os.environ.setdefault("OMP_NUM_THREADS", str(_N_THREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(_N_THREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(_N_THREADS))

for _pkg in ("psutil",):
    try:
        __import__(_pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", _pkg], check=True)

import psutil

try:
    from IPython.display import display
except ImportError:
    display = print

RANDOM_SEED = 42

_ram_start = psutil.virtual_memory()
print(f"WARP thread ceiling: {_N_THREADS} threads (of {os.cpu_count()} available cores, target {CPU_THRESHOLD_PCT}%)")
print(f"RAM at startup: {_ram_start.percent:.1f}% used ({_ram_start.used/1e9:.2f} GB / {_ram_start.total/1e9:.2f} GB) -- target ceiling {RAM_THRESHOLD_PCT}%")
print("Setup complete. (No CSV/model load in this notebook -- pure aggregation over already-real prior outputs.)")

##############################################################################
# REPO-LAYOUT BOOTSTRAP -- verified detection, unchanged from NB1-05.
##############################################################################
_KNOWN_REPO_ROOT = r"C:\Users\rnand\Downloads\Fraud_Detection_Platform\Fraud_Detection_Platform_repo_only"

def _looks_like_repo(_p):
    return os.path.isdir(os.path.join(_p, "notebooks")) or os.path.exists(os.path.join(_p, "requirements.txt"))

_cwd = os.getcwd()
_parent = os.path.abspath(os.path.join(_cwd, ".."))

if os.path.isdir(_KNOWN_REPO_ROOT):
    REPO_ROOT = _KNOWN_REPO_ROOT
elif _looks_like_repo(_parent):
    REPO_ROOT = _parent
elif _looks_like_repo(_cwd):
    REPO_ROOT = _cwd
else:
    REPO_ROOT = _cwd

_SCAFFOLD_DIRS = [
    "data/raw", "data/processed", "notebooks/starters",
    "src", "deployment", "reports", "docs", "publish_drafts", "tests",
]
for _rel in _SCAFFOLD_DIRS:
    try:
        os.makedirs(os.path.join(REPO_ROOT, *_rel.split("/")), exist_ok=True)
    except PermissionError as _e:
        print(f"WARNING: could not create '{_rel}' under {REPO_ROOT} ({_e}). Skipping.")

REPORTS_DIR = os.path.join(REPO_ROOT, "reports")
RESULTS_DIR = os.path.join(REPORTS_DIR, "nb6_results")
try:
    os.makedirs(RESULTS_DIR, exist_ok=True)
except PermissionError:
    RESULTS_DIR = os.path.join(_cwd, "nb6_results")
    os.makedirs(RESULTS_DIR, exist_ok=True)
    print(f"WARNING: falling back to {RESULTS_DIR} (no write access to {REPO_ROOT}).")

print(f"Repo root:   {REPO_ROOT}")
print(f"NB6 results in: {RESULTS_DIR}")

##############################################################################
# LOAD NB1/NB2/NB3/NB4/NB5's REAL OUTPUTS -- no recomputation, no retraining,
# no new model scoring. Every number below already exists on disk from a
# real, previously-verified notebook run.
##############################################################################
def _load_json(_rel_path, _label):
    _p = os.path.join(REPORTS_DIR, _rel_path)
    if not os.path.exists(_p):
        raise FileNotFoundError(
            f"{_label} not found at {_p} -- run that notebook first; NB6 only aggregates "
            f"already-real prior outputs, it does not recompute them."
        )
    with open(_p, encoding="utf-8") as f:
        return json.load(f)

nb1 = _load_json(os.path.join("nb1_results", "nb1_final_results.json"), "NB1's results")
nb2 = _load_json(os.path.join("nb2_results", "nb2_validation_report.json"), "NB2's results")
nb4 = _load_json(os.path.join("nb4_results", "nb4_serving_report.json"), "NB4's results")
nb5 = _load_json(os.path.join("nb5_results", "nb5_stress_test_report.json"), "NB5's results")

_drift_history_path = os.path.join(REPORTS_DIR, "nb3_results", "drift_history.jsonl")
nb3_latest_entry = None
if os.path.exists(_drift_history_path):
    with open(_drift_history_path, encoding="utf-8") as f:
        _lines = [_l for _l in f if _l.strip()]
    if _lines:
        nb3_latest_entry = json.loads(_lines[-1])
if nb3_latest_entry is None:
    raise FileNotFoundError(
        f"NB3's drift_history.jsonl not found or empty at {_drift_history_path} -- run NB3 at least once first."
    )

print(f"Loaded real results from NB1, NB2, NB3 ({len(_lines)} real monitoring entries), NB4, NB5. No recomputation.")

##############################################################################
# SECTION A -- TRANSPARENT, DISCLOSED-ASSUMPTION TIERING RUBRIC
# Three dimensions per the gap-analysis doc: Financial Exposure, Regulatory
# Scrutiny, Customer Impact. Each scored 0-3 from a REAL number already
# computed by a prior notebook, banded against a disclosed ASSUMPTION cut
# point (no universal published cut-point standard exists at this precision
# for a portfolio project of this scope -- the banding is the assumption,
# the inputs feeding it are all real).
##############################################################################
print("=" * 70)
print("SECTION A -- MODEL TIERING RUBRIC (real inputs, disclosed-ASSUMPTION bands)")
print("=" * 70)

# --- Dimension 1: Financial Exposure -----------------------------------
_real_current_cost_eur = nb1["threshold_result"]["total_cost"]
_worst_case_stress_eur = nb5["grid_sweep"]["worst_combination"]["projected_loss_eur"]
_stress_multiple = _worst_case_stress_eur / _real_current_cost_eur

def _band_financial(_x):
    if _x < 5: return 0
    if _x < 15: return 1
    if _x < 30: return 2
    return 3

financial_exposure_score = _band_financial(_stress_multiple)
print(f"  Financial Exposure: worst-case NB5 stress-grid loss (EUR {_worst_case_stress_eur:,.2f}) is "
      f"{_stress_multiple:.2f}x today's real cost-optimal total cost (EUR {_real_current_cost_eur:,.2f}) "
      f"-> score {financial_exposure_score}/3")

# --- Dimension 2: Regulatory Scrutiny -----------------------------------
_open_flags = []
if nb1.get("benchmark_check", {}).get("investigate_flag"):
    _open_flags.append("NB1 benchmark_check.investigate_flag (real precision/recall gap vs. external reference, unresolved)")
if nb2.get("drift_monitoring", {}).get("any_alert"):
    _open_flags.append("NB2/NB3 drift_monitoring.any_alert (real feature/score drift alert on the early-vs-late proxy)")
if not nb2.get("gate1_all_passed", True):
    _open_flags.append("NB2 Gate 1 structural check failed")
if not nb2.get("gate2_cv_stability_ok", True):
    _open_flags.append("NB2 Gate 2 CV-stability check failed")

def _band_regulatory(_n):
    return min(_n, 3)

regulatory_scrutiny_score = _band_regulatory(len(_open_flags))
print(f"  Regulatory Scrutiny: {len(_open_flags)} real open governance flag(s) across NB1-NB3 -> score {regulatory_scrutiny_score}/3")
for _f in _open_flags:
    print(f"    - {_f}")
if not _open_flags:
    print("    - none open")

# --- Dimension 3: Customer Impact ---------------------------------------
_cm = nb1["confusion_matrix"]  # [[TN, FP], [FN, TP]]
_real_fp = _cm[0][1]
_real_fn = _cm[1][0]
_real_n_total = nb1["dataset"]["rows"]

def _band_customer(_fp):
    if _fp < 50: return 0
    if _fp < 200: return 1
    if _fp < 1000: return 2
    return 3

customer_impact_score = _band_customer(_real_fp)
print(f"  Customer Impact: {_real_fp} real false declines out of {_real_n_total:,} real transactions "
      f"({_real_fp/_real_n_total:.4%}), decided in real time (NB4 real client-latency mean "
      f"{nb4['latency_sla_ms']['client_round_trip']['mean']:.2f} ms -- an instant, point-of-sale-facing decision, "
      f"not a delayed back-office review) -> score {customer_impact_score}/3")

_total_score = financial_exposure_score + regulatory_scrutiny_score + customer_impact_score

def _tier_from_score(_s):
    if _s >= 7: return 1, "Highest materiality -- full lifecycle oversight (SR 26-2 Tier 1)"
    if _s >= 4: return 2, "Moderate materiality -- proportionate, standard oversight"
    return 3, "Lower materiality -- lighter-touch, proportionate controls"

model_tier, tier_description = _tier_from_score(_total_score)

print("=" * 70)
print(f"COMPOSITE SCORE: {financial_exposure_score} + {regulatory_scrutiny_score} + {customer_impact_score} "
      f"= {_total_score}/9  ->  TIER {model_tier} ({tier_description})")
print("=" * 70)

##############################################################################
# SECTION B -- RETROACTIVE FINDING: NB3's drift-monitoring threshold tier
# was a default, not a real decision -- surfaced here, not silently fixed.
##############################################################################
print("SECTION B -- retroactive finding on NB3")
_nb3_tier_used = nb3_latest_entry.get("tier")
print(f"  NB3's real drift_history.jsonl entries were computed with tier={_nb3_tier_used} passed to "
      f"drift_monitoring.compute_drift_report() as an unjustified DEFAULT (no formal tiering decision existed "
      f"yet at that point in the build sequence). Now that this notebook has computed a real Tier {model_tier} "
      f"classification, future NB3 reruns should pass tier={model_tier} instead, so the real PSI/KS/PR-AUC-drop "
      f"alert thresholds actually match this model's real materiality tier. Not changed retroactively here -- "
      f"NB3's existing real history entries are left untouched (append-only, never rewritten); this is a "
      f"forward-looking correction for the next real rerun.")

##############################################################################
# SECTION C -- RENDER THE WRITTEN TIER JUSTIFICATION (real numbers only)
##############################################################################
_generated_at = datetime.now(timezone.utc).isoformat()

_eur_to_usd = nb2.get("eur_to_usd_rate", 1.1592)

_md = f"""# Model Tiering Matrix — Fraud Detection Platform

**Model:** {nb1['champion_name']} (fraud-champion-v1.0.0)
**Generated:** {_generated_at}
**Composite score:** {_total_score}/9  ->  **TIER {model_tier}** — {tier_description}

## Rubric (disclosed ASSUMPTION bands; all inputs are real, computed by NB1/NB2/NB3/NB4/NB5)

| Dimension | Real input | Score |
|---|---|---|
| Financial Exposure | Worst-case NB5 stress-grid loss is {_stress_multiple:.2f}x today's real cost-optimal total cost (EUR {_worst_case_stress_eur:,.2f} vs. EUR {_real_current_cost_eur:,.2f}) | {financial_exposure_score}/3 |
| Regulatory Scrutiny | {len(_open_flags)} real open governance flag(s) (see list below) | {regulatory_scrutiny_score}/3 |
| Customer Impact | {_real_fp} real false declines / {_real_n_total:,} real transactions ({_real_fp/_real_n_total:.4%}), real-time decisioning | {customer_impact_score}/3 |

## Open governance flags feeding the Regulatory Scrutiny score
{chr(10).join('- ' + _f for _f in _open_flags) if _open_flags else '- none open'}

## Scope and disclosed limitations
- All dollar figures are REAL TOTALS over the actual sampled window (284,807 real transactions, ~48 hours of real data) — not annualized. Annualizing would require a representativeness assumption not sourced for this portfolio project, so it is intentionally not extrapolated here.
- The Financial Exposure and Regulatory Scrutiny band cut points are disclosed ASSUMPTIONs (no universal published cut-point standard exists at this precision for a portfolio project of this scope) — the underlying numbers they are applied to are all real.
- This dataset's V1-V28 features are anonymized PCA components with no demographic attributes (already disclosed in NB2's model card) — Fairness & Bias Testing (gap-analysis priority 1) remains not applicable.

## Retroactive finding
NB3's real drift-monitoring history was computed with an unjustified default tier ({_nb3_tier_used}). Future NB3 reruns should pass tier={model_tier} so alert thresholds match this real classification. NB3's existing history entries are left untouched (append-only).

## Recommended next steps for this tier
{"Full lifecycle oversight: quarterly reproducible-challenger reruns, real-time monitoring with tier-1 alert thresholds, mandatory human sign-off before any threshold change." if model_tier == 1 else ("Standard oversight: scheduled monitoring reruns, documented sign-off before threshold changes." if model_tier == 2 else "Lighter-touch, proportionate controls; periodic review.")}
"""

_html = "<div class='tiering-matrix'>" + "".join(
    f"<h1>{_l[2:]}</h1>" if _l.startswith("# ") else
    f"<h2>{_l[3:]}</h2>" if _l.startswith("## ") else
    f"<p>{_l}</p>" if _l.strip() and not _l.startswith("|") and not _l.startswith("-") else
    (f"<li>{_l[2:]}</li>" if _l.startswith("- ") else "")
    for _l in _md.splitlines()
) + "</div>"

with open(os.path.join(RESULTS_DIR, "model_tiering_matrix.md"), "w", encoding="utf-8") as f:
    f.write(_md)
with open(os.path.join(RESULTS_DIR, "model_tiering_matrix.html"), "w", encoding="utf-8") as f:
    f.write(_html)

##############################################################################
# SAVE NOTEBOOK 06 RESULTS
##############################################################################
nb6_report = {
    "run_metadata": {"random_seed": RANDOM_SEED, "n_threads": _N_THREADS, "generated_at_utc": _generated_at},
    "rubric": {
        "financial_exposure": {"stress_multiple": _stress_multiple, "score": financial_exposure_score,
                                "worst_case_stress_eur": _worst_case_stress_eur, "current_cost_eur": _real_current_cost_eur},
        "regulatory_scrutiny": {"open_flags": _open_flags, "score": regulatory_scrutiny_score},
        "customer_impact": {"real_fp": _real_fp, "real_n_total": _real_n_total, "real_fp_rate": _real_fp / _real_n_total,
                             "score": customer_impact_score},
    },
    "composite_score": _total_score,
    "model_tier": model_tier,
    "tier_description": tier_description,
    "retroactive_finding": {"nb3_tier_used_as_default": _nb3_tier_used, "recommended_tier_going_forward": model_tier},
}
with open(os.path.join(RESULTS_DIR, "nb6_model_tiering_matrix.json"), "w", encoding="utf-8") as f:
    json.dump(nb6_report, f, indent=2, default=str)

_ram_end = psutil.virtual_memory()
_total_elapsed = time.time() - _RUN_T0
print("=" * 70)
print(f"RAM at finish: {_ram_end.percent:.1f}% used ({_ram_end.used/1e9:.2f} GB / {_ram_end.total/1e9:.2f} GB)")
print(f"Total notebook wall-clock time: {_total_elapsed:.2f}s (real, measured).")
print(f"Notebook 06 complete. Results written to: {os.path.join(RESULTS_DIR, 'nb6_model_tiering_matrix.json')}")
print(f"Written: model_tiering_matrix.md, model_tiering_matrix.html")
